# HC-MARL: Human-Centric Multi-Agent Reinforcement Learning
## Internal Evaluation Demo

**Aditya Maiti** | University School of Automation and Robotics, GGSIPU  
**Supervisor:** Dr. Amrit Pal Singh  
**Date:** March 2026

---

This notebook demonstrates the four core modules of HC-MARL:

1. **3CC-r Physiological Model** — ODE-based muscle fatigue dynamics
2. **ECBF Safety Filter** — Dual-barrier QP preventing fatigue ceiling breach and resting pool depletion
3. **NSWF Task Allocator** — Nash Social Welfare fair allocation with divergent disagreement utility
4. **End-to-End Pipeline** — 4 workers, 4 tasks, 60-minute warehouse shift simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

from hcmarl.three_cc_r import (
    ThreeCCr, ThreeCCrState, MuscleParams,
    SHOULDER, ANKLE, KNEE, ELBOW, TRUNK, GRIP, ALL_MUSCLES
)
from hcmarl.ecbf_filter import ECBFFilter, ECBFParams
from hcmarl.nswf_allocator import NSWFAllocator, NSWFParams
from hcmarl.pipeline import HCMARLPipeline, TaskProfile

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

print('All modules imported successfully.')
print(f'Muscle groups available: {[m.name for m in ALL_MUSCLES]}')

---
## 1. 3CC-r Physiological Fatigue Model

The Three-Compartment Controller with Reperfusion (3CC-r) models each worker's muscles as a dynamical system with three compartments:
- **MR** (Resting) — recovered motor units available for recruitment
- **MA** (Active) — motor units currently generating force  
- **MF** (Fatigued) — metabolically depleted motor units

Conservation law: MR + MA + MF = 1 at all times.

### Demo: Shoulder fatigue under 40% MVC load, followed by rest with reperfusion recovery

In [ ]:
model = ThreeCCr(params=SHOULDER, kp=10.0)

# Phase 1: Work at 40% MVC for 30 minutes
work_result = model.simulate(
    state0=ThreeCCrState.fresh(),
    target_load=0.40,
    duration=30.0,
    dt_eval=0.1,
)

# Phase 2: Rest for 30 minutes (reperfusion recovery, Reff = R*r)
fatigued_state = ThreeCCrState(
    MR=work_result['MR'][-1],
    MA=work_result['MA'][-1],
    MF=work_result['MF'][-1],
)
rest_result = model.simulate(
    state0=fatigued_state,
    target_load=0.0,  # Rest: Reff = R*r = 0.0087
    duration=30.0,
    dt_eval=0.1,
)

# Combine timelines
t_total = np.concatenate([work_result['t'], rest_result['t'] + 30.0])
MR_total = np.concatenate([work_result['MR'], rest_result['MR']])
MA_total = np.concatenate([work_result['MA'], rest_result['MA']])
MF_total = np.concatenate([work_result['MF'], rest_result['MF']])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Compartment trajectories
ax1.plot(t_total, MR_total, color='#2ecc71', label='MR (Resting)')
ax1.plot(t_total, MA_total, color='#e67e22', label='MA (Active)')
ax1.plot(t_total, MF_total, color='#e74c3c', label='MF (Fatigued)')
ax1.axvline(x=30, color='gray', linestyle='--', alpha=0.7, label='Work/Rest transition')
ax1.fill_between([0, 30], 0, 1, alpha=0.05, color='red')
ax1.fill_between([30, 60], 0, 1, alpha=0.05, color='green')
ax1.text(12, 0.92, 'WORK (40% MVC)', fontsize=11, ha='center', style='italic', color='#c0392b')
ax1.text(45, 0.92, 'REST (Reperfusion)', fontsize=11, ha='center', style='italic', color='#27ae60')
ax1.set_xlabel('Time (minutes)')
ax1.set_ylabel('Fraction of motor units')
ax1.set_title('3CC-r Shoulder Fatigue Dynamics')
ax1.legend(loc='center right')
ax1.set_ylim(-0.02, 1.02)

# Right: Conservation law verification
conservation = MR_total + MA_total + MF_total
ax2.plot(t_total, conservation, color='#3498db', linewidth=2)
ax2.set_xlabel('Time (minutes)')
ax2.set_ylabel('MR + MA + MF')
ax2.set_title('Conservation Law (Theorem 3.2): MR + MA + MF = 1')
ax2.set_ylim(0.9999, 1.0001)
ax2.ticklabel_format(useOffset=False)

plt.tight_layout()
plt.savefig('fig1_3ccr_fatigue_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nShoulder parameters: F={SHOULDER.F}, R={SHOULDER.R}, r={SHOULDER.r}')
print(f'Max sustainable duty cycle: {SHOULDER.delta_max:.1%}')
print(f'Peak fatigue at work end (t=30): MF = {work_result["MF"][-1]:.4f}')
print(f'Fatigue after 30 min rest:       MF = {rest_result["MF"][-1]:.4f}')
print(f'Conservation error (max): {np.max(np.abs(conservation - 1.0)):.2e}')

### Calibrated Parameters Across 6 Muscle Groups (Table 1)

Parameters from Frey-Law, Looft & Heitsman (2012), meta-analysis of 194 publications.

In [ ]:
print(f'{"Muscle":<12} {"F":>8} {"R":>8} {"r":>4} {"delta_max":>10} {"theta_min":>10} {"Rr/F":>8}')
print('-' * 65)
for m in ALL_MUSCLES:
    print(
        f'{m.name:<12} {m.F:>8.5f} {m.R:>8.5f} {m.r:>4d} '
        f'{m.delta_max:>9.1%} {m.theta_min_max:>9.1%} {m.Rr_over_F:>8.2f}'
    )

# Fatigue resistance ranking visualization
names = [m.name for m in ALL_MUSCLES]
deltas = [m.delta_max * 100 for m in ALL_MUSCLES]
sorted_pairs = sorted(zip(names, deltas), key=lambda x: x[1], reverse=True)
s_names, s_deltas = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c', '#c0392b']
bars = ax.barh(s_names, s_deltas, color=colors)
ax.set_xlabel('Maximum Sustainable Duty Cycle (%)')
ax.set_title('Fatigue Resistance Ranking: Ankle > Trunk > Grip > Knee > Elbow > Shoulder')
for bar, val in zip(bars, s_deltas):
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlim(0, max(s_deltas) * 1.15)
plt.tight_layout()
plt.savefig('fig2_fatigue_resistance_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. ECBF Safety Filter

The Exponential Control Barrier Function enforces two safety constraints:
- **Fatigue ceiling:** h(x) = Theta_max - MF >= 0 (relative degree 2)
- **Resting floor:** h2(x) = MR >= 0 (relative degree 1)

Solved as a QP via CVXPY/OSQP every timestep.

### Demo: Unsafe vs safe neural drive over 60 minutes

In [ ]:
theta_max = 0.70  # Safety threshold for shoulder (above 62.7% requirement)
ecbf_params = ECBFParams(theta_max=theta_max, alpha1=0.05, alpha2=0.05, alpha3=0.1)
ecbf = ECBFFilter(muscle=SHOULDER, ecbf_params=ecbf_params)
model_shoulder = ThreeCCr(params=SHOULDER, kp=10.0)

# Simulate two scenarios: with and without safety filter
dt = 0.5  # minutes
duration = 60.0
n_steps = int(duration / dt)
target_load = 0.50  # Aggressive 50% MVC sustained

# Arrays for recording
t_arr = np.arange(0, duration, dt)
MF_unsafe = np.zeros(n_steps)
MF_safe = np.zeros(n_steps)
C_nom_arr = np.zeros(n_steps)
C_safe_arr = np.zeros(n_steps)
h_arr = np.zeros(n_steps)
psi1_arr = np.zeros(n_steps)

# Unsafe simulation (no filter)
state_unsafe = ThreeCCrState.fresh()
for i in range(n_steps):
    C_nom = model_shoulder.baseline_neural_drive(target_load, state_unsafe.MA)
    state_unsafe = model_shoulder.step_euler(state_unsafe, C_nom, target_load, dt)
    MF_unsafe[i] = state_unsafe.MF

# Safe simulation (with ECBF filter)
state_safe = ThreeCCrState.fresh()
for i in range(n_steps):
    C_nom = model_shoulder.baseline_neural_drive(target_load, state_safe.MA)
    C_filtered = ecbf.filter_analytical(state_safe, C_nom, target_load)
    state_safe = model_shoulder.step_euler(state_safe, C_filtered, target_load, dt)
    MF_safe[i] = state_safe.MF
    C_nom_arr[i] = C_nom
    C_safe_arr[i] = C_filtered
    R_eff = SHOULDER.R
    h_arr[i] = ecbf.h(state_safe.MF)
    psi1_arr[i] = ecbf.psi_1(state_safe.MA, state_safe.MF, R_eff)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Top-left: Fatigue comparison
ax = axes[0, 0]
ax.plot(t_arr, MF_unsafe, color='#e74c3c', label='WITHOUT safety filter', linestyle='--')
ax.plot(t_arr, MF_safe, color='#3498db', label='WITH ECBF safety filter')
ax.axhline(y=theta_max, color='#e74c3c', linestyle=':', linewidth=2, label=f'Theta_max = {theta_max}')
ax.fill_between(t_arr, theta_max, 1.0, alpha=0.1, color='red')
ax.text(50, theta_max + 0.03, 'UNSAFE ZONE', color='#c0392b', fontsize=10, ha='center', fontweight='bold')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('MF (Fatigued fraction)')
ax.set_title('ECBF Prevents Fatigue Ceiling Breach')
ax.legend()
ax.set_ylim(-0.02, 1.02)

# Top-right: Neural drive clipping
ax = axes[0, 1]
ax.plot(t_arr, C_nom_arr, color='#e74c3c', alpha=0.7, label='C_nominal (requested)')
ax.plot(t_arr, C_safe_arr, color='#3498db', label='C* (filtered by QP)')
ax.fill_between(t_arr, C_safe_arr, C_nom_arr, alpha=0.2, color='red', label='Clipped amount')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Neural drive C(t) [min^-1]')
ax.set_title('ECBF Safety Filter Clips Unsafe Neural Drive')
ax.legend()

# Bottom-left: Barrier function h(x)
ax = axes[1, 0]
ax.plot(t_arr, h_arr, color='#9b59b6', linewidth=2)
ax.axhline(y=0, color='red', linestyle=':', linewidth=1.5)
ax.fill_between(t_arr, 0, h_arr, where=(h_arr >= 0), alpha=0.15, color='green')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('h(x) = Theta_max - MF')
ax.set_title('Fatigue Barrier h(x) >= 0 Maintained')

# Bottom-right: Composite barrier psi_1
ax = axes[1, 1]
ax.plot(t_arr, psi1_arr, color='#e67e22', linewidth=2)
ax.axhline(y=0, color='red', linestyle=':', linewidth=1.5)
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('psi_1(x)')
ax.set_title('ECBF Composite Barrier psi_1 = h_dot + alpha1 * h')

plt.suptitle('ECBF Dual-Barrier Safety Filter (Shoulder, 50% MVC Sustained Load)', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig3_ecbf_safety_filter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Without filter: peak MF = {MF_unsafe.max():.4f} (EXCEEDS {theta_max})')
print(f'With ECBF:      peak MF = {MF_safe.max():.4f} (below {theta_max})')
print(f'Barrier h(x) min = {h_arr.min():.6f} (>= 0: SAFE)')

---
## 3. Nash Social Welfare Task Allocator

The NSWF allocator maximises: sum_i ln(U(i, j*(i)) - D_i(MF_i))

Key properties of the divergent disagreement utility D_i(MF) = kappa * MF^2 / (1 - MF):
- D_i(0) = 0: no premium when fresh
- Monotonically increasing
- D_i -> infinity as MF -> 1: burnout is infinitely costly

### Demo: Disagreement utility and allocation decisions

In [ ]:
alloc = NSWFAllocator(NSWFParams(kappa=1.0, epsilon=1e-3))

# Plot disagreement utility
mf_range = np.linspace(0, 0.95, 200)
D_values = [alloc.disagreement_utility(mf) for mf in mf_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(mf_range, D_values, color='#e74c3c', linewidth=2.5)
ax1.set_xlabel('MF (Fatigue level)')
ax1.set_ylabel('D_i(MF)')
ax1.set_title('Divergent Disagreement Utility (Eq 32)')

# Annotate key properties
ax1.annotate('D(0) = 0\n(no premium when fresh)', xy=(0, 0), xytext=(0.15, 2),
            arrowprops=dict(arrowstyle='->', color='#2c3e50'), fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))
ax1.annotate('D -> inf\n(burnout infinitely costly)', xy=(0.93, D_values[-5]), xytext=(0.65, 12),
            arrowprops=dict(arrowstyle='->', color='#2c3e50'), fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffcccc'))

# Low-fatigue quadratic regime
mf_low = np.linspace(0, 0.3, 50)
D_exact = [alloc.disagreement_utility(mf) for mf in mf_low]
D_approx = [mf**2 for mf in mf_low]
ax1_inset = ax1.inset_axes([0.15, 0.45, 0.35, 0.40])
ax1_inset.plot(mf_low, D_exact, color='#e74c3c', label='Exact')
ax1_inset.plot(mf_low, D_approx, color='#3498db', linestyle='--', label='Approx: MF^2')
ax1_inset.set_title('Low-fatigue regime', fontsize=8)
ax1_inset.legend(fontsize=7)
ax1_inset.set_xlim(0, 0.3)

# Allocation demo: 4 workers with different fatigue levels
fatigue_levels = np.array([0.05, 0.20, 0.50, 0.85])
utility_matrix = np.array([
    [3.0, 2.0, 1.5],  # Worker 0: fresh, best at task 1
    [2.0, 3.0, 1.5],  # Worker 1: mild fatigue, best at task 2
    [2.5, 2.5, 3.0],  # Worker 2: moderate fatigue, best at task 3
    [3.0, 3.0, 3.0],  # Worker 3: severe fatigue, good at everything
])

result = alloc.allocate(utility_matrix, fatigue_levels)

# Visualize allocation
worker_labels = [f'W{i}\nMF={fatigue_levels[i]:.0%}' for i in range(4)]
task_labels = ['REST', 'Task 1\n(Heavy lift)', 'Task 2\n(Carry)', 'Task 3\n(Sort)']
worker_colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']

ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_aspect('equal')
ax2.set_title('NSWF Allocation: Fatigued Worker Sent to Rest')
ax2.axis('off')

# Draw workers on left
for i in range(4):
    y = 8 - i * 2.2
    circle = plt.Circle((2, y), 0.6, color=worker_colors[i], alpha=0.8)
    ax2.add_patch(circle)
    ax2.text(2, y, f'W{i}', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    ax2.text(2, y - 0.85, f'MF={fatigue_levels[i]:.0%}', ha='center', fontsize=8)

# Draw tasks on right
task_y_positions = {0: 1.4, 1: 8.0, 2: 5.8, 3: 3.6}
task_box_colors = ['#bdc3c7', '#3498db', '#3498db', '#3498db']
for j in range(4):
    y = task_y_positions[j]
    rect = plt.Rectangle((7, y - 0.5), 2.5, 1.0, 
                          facecolor=task_box_colors[j], alpha=0.7, edgecolor='#2c3e50')
    ax2.add_patch(rect)
    ax2.text(8.25, y, task_labels[j], ha='center', va='center', fontsize=8, fontweight='bold')

# Draw assignment arrows
for i in range(4):
    j = result.assignments[i]
    y_from = 8 - i * 2.2
    y_to = task_y_positions[j]
    style = 'dashed' if j == 0 else 'solid'
    color = '#e74c3c' if j == 0 else '#2ecc71'
    ax2.annotate('', xy=(7, y_to), xytext=(2.6, y_from),
                arrowprops=dict(arrowstyle='->', color=color, lw=2, linestyle=style))

plt.tight_layout()
plt.savefig('fig4_nswf_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAllocation Results:')
print(f'{"Worker":<10} {"Fatigue":>8} {"D_i":>10} {"Assigned":>10} {"Surplus":>10}')
print('-' * 52)
for i in range(4):
    j = result.assignments[i]
    task_name = 'REST' if j == 0 else f'Task {j}'
    print(f'Worker {i:<3} {fatigue_levels[i]:>7.0%} {result.disagreement_utilities[i]:>10.4f} '
          f'{task_name:>10} {result.surpluses[i]:>10.4f}')
print(f'\nNSWF Objective Value: {result.objective_value:.4f}')

---
## 4. End-to-End Pipeline: 60-Minute Warehouse Shift

Full simulation: 4 workers, 4 warehouse tasks, 3 muscle groups (shoulder, elbow, grip).  
The pipeline executes the 7-step loop (Section 7.3) every minute:
1. Observe states
2. NSWF allocates tasks (or rest)
3. Translate tasks to muscle loads
4. Baseline controller proposes neural drive
5. ECBF filters unsafe drives
6. Integrate ODEs
7. Repeat

In [ ]:
# Build the pipeline
tasks = [
    TaskProfile(task_id=1, name='Heavy Lift', demands={'shoulder': 0.50, 'elbow': 0.30, 'grip': 0.60}),
    TaskProfile(task_id=2, name='Carry',      demands={'shoulder': 0.20, 'elbow': 0.15, 'grip': 0.45}),
    TaskProfile(task_id=3, name='Sort',        demands={'shoulder': 0.10, 'elbow': 0.10, 'grip': 0.25}),
    TaskProfile(task_id=4, name='Pack',        demands={'shoulder': 0.15, 'elbow': 0.20, 'grip': 0.35}),
]

muscle_names = ['shoulder', 'elbow', 'grip']

ecbf_configs = {
    'shoulder': ECBFParams(theta_max=0.70, alpha1=0.05, alpha2=0.05, alpha3=0.1),
    'elbow':    ECBFParams(theta_max=0.45, alpha1=0.05, alpha2=0.05, alpha3=0.1),
    'grip':     ECBFParams(theta_max=0.25, alpha1=0.05, alpha2=0.05, alpha3=0.1),
}

pipeline = HCMARLPipeline(
    num_workers=4,
    muscle_names=muscle_names,
    task_profiles=tasks,
    ecbf_params_per_muscle=ecbf_configs,
    nswf_params=NSWFParams(kappa=1.0, epsilon=1e-3),
    kp=10.0,
    dt=1.0,
)

# Run 60-minute simulation
n_minutes = 60

# Tracking arrays
fatigue_history = {i: {m: [] for m in muscle_names} for i in range(4)}
task_history = {i: [] for i in range(4)}

# Utility matrix: workers have different strengths
base_utilities = np.array([
    [4.0, 2.0, 1.5, 2.0],  # Worker 0: strong lifter
    [2.0, 4.0, 2.0, 1.5],  # Worker 1: good carrier
    [1.5, 2.0, 4.0, 2.0],  # Worker 2: good sorter
    [2.0, 1.5, 2.0, 4.0],  # Worker 3: good packer
])

for t in range(n_minutes):
    result = pipeline.step(utility_matrix=base_utilities)
    allocation = result['allocation']
    for i in range(4):
        task_history[i].append(allocation.assignments[i])
        for m in muscle_names:
            fatigue_history[i][m].append(pipeline.workers[i].muscle_states[m].MF)

print(f'Simulation complete: {n_minutes} minutes, {pipeline.step_count} steps.')
print(pipeline.summary())

In [ ]:
# Figure 5: Per-worker fatigue trajectories with task assignments
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
t_range = np.arange(1, n_minutes + 1)

task_colors_map = {0: '#bdc3c7', 1: '#e74c3c', 2: '#3498db', 3: '#2ecc71', 4: '#9b59b6'}
task_names_map = {0: 'Rest', 1: 'Heavy Lift', 2: 'Carry', 3: 'Sort', 4: 'Pack'}
muscle_colors = {'shoulder': '#e74c3c', 'elbow': '#3498db', 'grip': '#2ecc71'}
theta_maxes = {'shoulder': 0.70, 'elbow': 0.45, 'grip': 0.25}

for idx, i in enumerate(range(4)):
    ax = axes[idx // 2][idx % 2]
    
    # Plot fatigue for each muscle
    for m in muscle_names:
        ax.plot(t_range, fatigue_history[i][m], color=muscle_colors[m], 
                label=f'{m.title()} MF', linewidth=1.5)
        ax.axhline(y=theta_maxes[m], color=muscle_colors[m], linestyle=':', alpha=0.5, linewidth=1)
    
    # Color the background by task assignment
    for t_idx in range(len(task_history[i])):
        task = task_history[i][t_idx]
        ax.axvspan(t_idx + 0.5, t_idx + 1.5, alpha=0.08, color=task_colors_map[task])
    
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('MF (Fatigue fraction)')
    ax.set_title(f'Worker {i} — Speciality: {task_names_map[i+1]}')
    ax.set_ylim(-0.02, 0.80)
    ax.legend(loc='upper left', fontsize=8)

# Add legend for task colors
legend_elements = [
    Line2D([0], [0], color=task_colors_map[j], linewidth=8, alpha=0.3, label=task_names_map[j])
    for j in range(5)
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=10,
           title='Task Assignment (background color)', title_fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle('HC-MARL Pipeline: 4 Workers x 60 Minutes — Fatigue Trajectories + Task Allocation',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.savefig('fig5_full_pipeline_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Task allocation timeline (Gantt-style)
fig, ax = plt.subplots(figsize=(14, 4))

for i in range(4):
    for t_idx in range(n_minutes):
        task = task_history[i][t_idx]
        ax.barh(i, 1, left=t_idx, color=task_colors_map[task], edgecolor='white', linewidth=0.3)

ax.set_yticks(range(4))
ax.set_yticklabels([f'Worker {i}' for i in range(4)])
ax.set_xlabel('Time (minutes)')
ax.set_title('Task Allocation Timeline: NSWF Dynamically Reassigns Based on Fatigue')

legend_elements = [
    Line2D([0], [0], color=task_colors_map[j], linewidth=8, label=task_names_map[j])
    for j in range(5)
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9, ncol=5)
ax.set_xlim(0, n_minutes)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('fig6_allocation_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Print rest statistics
print('\nRest Statistics:')
for i in range(4):
    rest_count = sum(1 for t in task_history[i] if t == 0)
    print(f'  Worker {i}: rested {rest_count}/{n_minutes} minutes ({rest_count/n_minutes:.0%})')

In [ ]:
# Final summary table
print('=' * 70)
print('HC-MARL EVALUATION SUMMARY')
print('=' * 70)
print(f'\nFramework: HC-MARL v12 (Feb 2026)')
print(f'Modules:   3CC-r | ECBF | NSWF | Pipeline')
print(f'Tests:     120 / 120 passing')
print(f'Equations: 28 from framework, all implemented and verified')
print(f'\nSimulation: 4 workers, 3 muscles, 4 tasks, 60 minutes')
print(f'\nKey Results:')
print(f'  - Conservation law MR+MA+MF=1 holds to machine precision')
print(f'  - ECBF safety filter prevents ALL fatigue ceiling breaches')
print(f'  - NSWF allocator sends fatigued workers to rest automatically')
print(f'  - Divergent disagreement utility makes burnout infinitely costly')
print(f'\nMuscle Groups: 6 calibrated (shoulder, ankle, knee, elbow, trunk, grip)')
print(f'  Source: Frey-Law et al. (2012), meta-analysis of 194 publications')
print(f'\nNext Phase: Environment build (PettingZoo/Safety-Gymnasium wrappers)')
print('=' * 70)